In [ ]:
export MODEL="Your model here"
# export MODEL="unsloth/gemma-4-E4B-it-GGUF:UD-Q6_K_XL"
# to set sudo sysctl iogpu.wired_limit_mb=14800. Needs to be set on termimal outside for pure GPU
sysctl iogpu.wired_limit_mb # to check

In [ ]:
llama-fit-params -hf $MODEL # This is just for downloading and some initial values to probe

In [14]:
# Configurable batch sizes to test to fit context
# Though it looks like batch sizes have no effects 
# as per README 
# "Increasing this value above the value of the physical batch size may improve prompt processing performance 
# when using multiple GPUs with pipeline parallelism. Default: `2048`" 

BATCH_SIZES="1024 2048" # 2 args so we can see the diff, shoud not make a difference  
UBATCH_SIZES="64 128 256 512"
FITT="256 512"

echo "Testing different batch/ubatch/fitt combinations for ${MODEL}..." 
  
for ub in $UBATCH_SIZES; do  
    for b in $BATCH_SIZES; do
        for ft in $FITT; do
            # Get fitted parameters  
            #OUTPUT=$(llama-fit-params -hf $MODEL -b $b -ub $ub | grep -o '\-c [0-9]* \-ngl [0-9]*')  \
            
            # need to add fitt for nvidia
            OUTPUT=$(llama-fit-params -hf $MODEL -b $b -ub $ub -fitt $ft 2>&1)
            
            #echo "Raw output: $OUTPUT"  # Debug line  
            
            # Extract context and GPU layers more robustly  
            CTX=$(echo "$OUTPUT" | grep -o '\-c [0-9]*' | awk '{print $2}')  
            NGL=$(echo "$OUTPUT" | grep -o '\-ngl -\?[0-9]*' | awk '{print $2}')  
            
            if [ ! -z "$CTX" ] && [ ! -z "$NGL" ]; then  
                echo "ub: $ub, b: $b, fitt: $ft, ctx: $CTX, ngl: $NGL"  
            else  
                echo "No valid parameters found"  
            fi 
        done
    done  
done 
  

Testing different batch/ubatch combinations...
b: 1024, ub: 128, ctx: 55040, ngl: -1
b: 2048, ub: 128, ctx: 55040, ngl: -1
b: 4096, ub: 128, ctx: 55040, ngl: -1
b: 1024, ub: 256, ctx: 48384, ngl: -1
b: 2048, ub: 256, ctx: 48384, ngl: -1
b: 4096, ub: 256, ctx: 48384, ngl: -1
b: 1024, ub: 512, ctx: 35328, ngl: -1
b: 2048, ub: 512, ctx: 35328, ngl: -1
b: 4096, ub: 512, ctx: 35328, ngl: -1
b: 1024, ub: 1024, ctx: 11264, ngl: -1
b: 2048, ub: 1024, ctx: 11264, ngl: -1
b: 4096, ub: 1024, ctx: 11264, ngl: -1


In [ ]:
# Pure GPU
echo "Benching ${MODEL}..." 
llama-bench -hf $MODEL -ngl 99 -t 8 -ub 128,256,512 -p 1000,2000 -n 256,512

In [ ]:
# batched bench seems the closer to llama-server as judged from the logs, thats' why it is here
# llama-batched-bench -hf $MODEL \
#     -ngl 99 --threads 8 -fa on \
#     -c 16000 \
#     -npp 1000 -ntg 1000 -npl 1

In [ ]:
llama-fit-params -hf $MODEL -fitt 512 # fitt leaves that around of memory in MiB in the target devices

In [ ]:
# echo "Staring llama-server for ${MODEL}..." 
# llama-server -hf $MODEL \
#     --threads-http 1 --fit on --threads 8 --no-mmproj --reasoning off \
#     --no-warmup -ngl -1 -np 1 --host 0.0.0.0 # -np 1 parameter is important

In [ ]:
# lv=3 gives out the same logs as llama-server
echo "Staring llama-cli for ${MODEL}..." 
llama-cli -lv 3 -hf $MODEL --threads 8 --no-mmproj --reasoning off \
    -ngl -1 -st -p "Who are you?"